In [4]:
import os
import sys
import pandas as pd
import numpy as np
# Sube un nivel desde 'notebook' y entra a 'src'
ruta_src = os.path.abspath(os.path.join("..", "src"))
if ruta_src not in sys.path:
    sys.path.append(ruta_src)
from feature_engineering import (build_quant_features,quant_features,)
from preprocessing import (temporal_split,fit_robust_params,transform_returns,)
input_train = pd.read_csv("../data/input_training.csv")
output_train = pd.read_csv("../data/output_training_gmEd6Zt.csv")

input_test = pd.read_csv("../data/input_test.csv")
output_test = pd.read_csv("../data/output_test_random.csv")

return_cols = [f"r{i}" for i in range(53)]

train = input_train.merge(
    output_train,
    on="ID",
    how="inner",
    validate="one_to_one"
)

train_dev, val_dev = temporal_split(train)
robust_params = fit_robust_params(train_dev,return_cols)
train_processed = transform_returns(train_dev,return_cols,robust_params)
val_processed = transform_returns(val_dev,return_cols,robust_params)

### Aqui tenemos:

$$train \longrightarrow \text{temporal split} \begin{cases} train\_dev \\ val\_dev \end{cases}$$

Donde `train_dev` y `val_dev` siguen siendo los retornos originales en bps.

Después:

$$train\_dev, val\_dev \longrightarrow \text{preprocessing 02} \begin{cases} train\_processed \\ val\_processed \end{cases}$$

Estos últimos contienen los r0...r52:

* **Robust-scaled** $\longrightarrow$ **clipped [-20,20]** $\longrightarrow$ **NaN=0**, además de las variables de missingness.

No se han construido las quant features de 03. Ese es el siguiente paso, pero se construyen desde `train_dev` y `val_dev`, no desde `train_processed`, porque `build_quant_features()` necesita los retornos en bps.


In [5]:
train_features, val_features, lower_bounds, upper_bounds = (build_quant_features(
        train_dev,
        val_dev,
        return_cols
    ))

#train_processed / val_processed
# → secuencia r0...r52 preprocesada

# train_features / val_features
# → quant features construidas en bps

### 1. Matrices de modelado

Para el primer baseline vamos a utilizar las quant features de 03, no los 53 retornos.

In [6]:
X_train_quant = train_features[quant_features].copy()
X_val_quant = val_features[quant_features].copy()

y_train = train_features["reod"].copy()
y_val = val_features["reod"].copy()

print("X_train:", X_train_quant.shape)
print("X_val:", X_val_quant.shape)

print("\nTarget train:")
print(y_train.value_counts(normalize=True).sort_index())

print("\nTarget validation:")
print(y_val.value_counts(normalize=True).sort_index())

print("\nNaNs:")
print("Train:", X_train_quant.isna().sum().sum())
print("Validation:", X_val_quant.isna().sum().sum())

X_train: (673751, 14)
X_val: (169548, 14)

Target train:
reod
-1    0.299269
 0    0.409567
 1    0.291164
Name: proportion, dtype: float64

Target validation:
reod
-1    0.306733
 0    0.421822
 1    0.271445
Name: proportion, dtype: float64

NaNs:
Train: 10909
Validation: 2534


In [7]:
#Para el baseline, 
#imputemos las quant features con la mediana aprendida exclusivamente en train:
from sklearn.impute import SimpleImputer

quant_imputer = SimpleImputer(strategy="median")

X_train_quant_imp = quant_imputer.fit_transform(X_train_quant)
X_val_quant_imp = quant_imputer.transform(X_val_quant)

print("NaNs train:", np.isnan(X_train_quant_imp).sum())
print("NaNs validation:", np.isnan(X_val_quant_imp).sum())

NaNs train: 0
NaNs validation: 0


In [ ]:
# Baseline 0 — Dummy Classifier
#Ya sabemos que predecir siempre la clase mayoritaria 0 
# debería dar aproximadamente 42.18% en validation.
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score

dummy = DummyClassifier(strategy="most_frequent")
dummy.fit(X_train_quant_imp, y_train)
y_pred_dummy = dummy.predict(X_val_quant_imp)
print("Dummy Classifier")
print(f"Accuracy:          {accuracy_score(y_val, y_pred_dummy):.4f}")
print(f"Balanced Accuracy: {balanced_accuracy_score(y_val, y_pred_dummy):.4f}")
print(f"Macro F1:          {f1_score(y_val, y_pred_dummy, average='macro'):.4f}")

#Por qué tres métricas?

#Accuracy: es la métrica principal del challenge.
#Balanced Accuracy: evita que nos engañe simplemente predecir mucho la clase 0.
#Macro F1: comprueba si realmente somos capaces de identificar las tres clases.

Dummy Classifier
Accuracy:          0.4218
Balanced Accuracy: 0.3333
Macro F1:          0.1978


### Análisis del Baseline y Próximos Pasos

El **Balanced Accuracy de 33.33%** y el **Macro F1 de 19.78%** confirman que el modelo actual no tiene capacidad predictiva real y se limita a predecir siempre la clase mayoritaria (0). 

Esto redefine nuestra métrica de éxito en el dataset de validación:
* **Referencia del challenge:** Se mencionaba un ~33% como rendimiento aleatorio.
* **Nuestra realidad:** La clase mayoritaria en validación representa el **42.18%**. 
* **Umbral mínimo:** Superar el 33% no es suficiente; cualquier modelo útil debe superar el **42.18%**.

$$Modelo\ Útil > 42.18\%$$

#### Siguiente Experimento: Regresión Logística
El próximo paso es entrenar una Regresión Logística utilizando las 14 *quant features*. Este experimento servirá para evaluar la naturaleza de las variables:
* Si supera el **42.18%**, confirmaremos que las variables contienen una señal lineal útil.
* Si no lo supera, validaremos que las relaciones son estrictamente no lineales, dejando la responsabilidad al posterior modelo basado en árboles (Tree-based models).


In [9]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score

logit = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(
        max_iter=1000,
        random_state=42
    ))
])

logit.fit(X_train_quant_imp, y_train)

y_pred_logit = logit.predict(X_val_quant_imp)

print("Logistic Regression")
print(f"Accuracy:          {accuracy_score(y_val, y_pred_logit):.4f}")
print(f"Balanced Accuracy: {balanced_accuracy_score(y_val, y_pred_logit):.4f}")
print(f"Macro F1:          {f1_score(y_val, y_pred_logit, average='macro'):.4f}")

Logistic Regression
Accuracy:          0.4546
Balanced Accuracy: 0.3934
Macro F1:          0.3577


### Decisión

Logistic Regression : baseline válido y supera al Dummy 

otra lectura importante: nuestras quant features sí contienen información predictiva conjunta fuera de muestra. Ya tenemos evidencia de ello incluso con un modelo lineal relativamente sencillo.

In [ ]:
# Baseline 2 — HistGradientBoosting
# como baseline no lineal. No hacemos tuning todavía; 
# queremos medir si capturar thresholds e interacciones mejora el 45.46% del Logistic.
from sklearn.ensemble import HistGradientBoostingClassifier

hgb = HistGradientBoostingClassifier(
    learning_rate=0.1,
    max_iter=100,
    max_leaf_nodes=31,
    random_state=42
)

hgb.fit(X_train_quant_imp, y_train)

y_pred_hgb = hgb.predict(X_val_quant_imp)

print("HistGradientBoosting")
print(f"Accuracy:          {accuracy_score(y_val, y_pred_hgb):.4f}")
print(f"Balanced Accuracy: {balanced_accuracy_score(y_val, y_pred_hgb):.4f}")
print(f"Macro F1:          {f1_score(y_val, y_pred_hgb, average='macro'):.4f}")

HistGradientBoosting
--------------------
Accuracy:          0.4687
Balanced Accuracy: 0.4254
Macro F1:          0.4144


| Modelo                   |   Accuracy | Balanced Acc. |   Macro F1 |
| ------------------------ | ---------: | ------------: | ---------: |
| Dummy                    |     42.18% |        33.33% |     19.78% |
| Logistic                 |     45.46% |        39.34% |     35.77% |
| **HistGradientBoosting** | **46.87%** |    **42.54%** | **41.44%** |


El siguiente checkpoint debe ser entender dónde está acertando y fallando HGB por clase mediante una confusion matrix / classification report. Eso nos dirá si el 46.87% proviene de una mejora equilibrada o si todavía existe una clase especialmente difícil.


In [11]:
# Diagnóstico por clase — HGB
from sklearn.metrics import classification_report, confusion_matrix

print(classification_report(
        y_val,
        y_pred_hgb,
        labels=[-1, 0, 1],
        digits=4
    )
)

cm = confusion_matrix(
    y_val,
    y_pred_hgb,
    labels=[-1, 0, 1]
)

cm

              precision    recall  f1-score   support

          -1     0.3794    0.3044    0.3378     52006
           0     0.5484    0.7414    0.6304     71519
           1     0.3407    0.2305    0.2750     46023

    accuracy                         0.4687    169548
   macro avg     0.4228    0.4254    0.4144    169548
weighted avg     0.4402    0.4687    0.4442    169548



array([[15829, 22784, 13393],
       [11357, 53021,  7141],
       [14536, 20878, 10609]])

In [ ]:
cm_norm = confusion_matrix(
    y_val,
    y_pred_hgb,
    labels=[-1, 0, 1],
    normalize="true"
)

cm_norm
#matriz normalizada por clase real, porque los conteos absolutos pueden engañarnos:

array([[0.30436873, 0.4381033 , 0.25752798],
       [0.15879696, 0.74135544, 0.09984759],
       [0.31584208, 0.45364274, 0.23051518]])

### Diagnóstico direccional — return_late_bps

- El retorno reciente observado antes de las 14:00 contiene información sobre si reod será -1, 0 o +1?

In [13]:
# Quintiles de return_late_bps aprendidos sólo en train
late_return_bins = train_features["return_late_bps"].quantile(
    [0, .2, .4, .6, .8, 1]
).values

late_return_bins[0] = -np.inf
late_return_bins[-1] = np.inf

train_features["late_return_quintile"] = pd.cut(
    train_features["return_late_bps"],
    bins=late_return_bins,
    labels=["Q1", "Q2", "Q3", "Q4", "Q5"],
    include_lowest=True
)

val_features["late_return_quintile"] = pd.cut(
    val_features["return_late_bps"],
    bins=late_return_bins,
    labels=["Q1", "Q2", "Q3", "Q4", "Q5"],
    include_lowest=True
)
train_late_direction = pd.crosstab(
    train_features["late_return_quintile"],
    train_features["reod"],
    normalize="index"
)

val_late_direction = pd.crosstab(
    val_features["late_return_quintile"],
    val_features["reod"],
    normalize="index"
)

print("TRAIN")
display(train_late_direction)

print("\nVALIDATION")
display(val_late_direction)

TRAIN


reod,-1,0,1
late_return_quintile,,,
Q1,0.325200,0.331352,0.343448
Q2,0.293139,0.422576,0.284285
Q3,0.232241,0.548793,0.218965
Q4,0.298110,0.421621,0.280269
Q5,0.347659,0.323488,0.328853



VALIDATION


reod,-1,0,1
late_return_quintile,,,
Q1,0.395994,0.322466,0.281540
Q2,0.309769,0.428397,0.261835
Q3,0.221900,0.568788,0.209312
Q4,0.275130,0.446972,0.277898
Q5,0.331932,0.344684,0.323384


### Late momentum consistency

Queremos distinguir entre dos trayectorias con el mismo `return_late_bps`:

* Una donde muchos intervalos recientes apoyan esa dirección.
* Otra donde el retorno neto proviene de uno o dos movimientos grandes.

Definimos:

$$LateMomentumConsistency = \frac{\#\{\text{retornos late con el mismo signo que } R_{late}\}}{\#\{\text{retornos late observados}\}}$$


In [14]:
def add_late_momentum_consistency(df, late_cols):

    late_return = df[late_cols].sum(axis=1, skipna=True)
    late_sign = np.sign(late_return)

    interval_signs = np.sign(df[late_cols])
    observed = df[late_cols].notna()

    same_direction = interval_signs.eq(late_sign, axis=0)

    n_late_obs = observed.sum(axis=1)

    df["late_momentum_consistency"] = np.where(
        n_late_obs > 0,
        (same_direction & observed).sum(axis=1) / n_late_obs,
        np.nan
    )

    return df


late_cols = [f"r{i}" for i in range(36, 53)]

train_features = add_late_momentum_consistency(
    train_features,
    late_cols
)

val_features = add_late_momentum_consistency(
    val_features,
    late_cols
)

train_features["late_momentum_consistency"].describe(
    percentiles=[.25, .50, .75, .90, .95, .99]
)

count    665302.000000
mean          0.450825
std           0.183682
min           0.000000
25%           0.333333
50%           0.470588
75%           0.529412
90%           0.647059
95%           0.705882
99%           1.000000
max           1.000000
Name: late_momentum_consistency, dtype: float64

In [15]:
consistency_bins = train_features[
    "late_momentum_consistency"
].quantile([0, .2, .4, .6, .8, 1]).values

consistency_bins[0] = -np.inf
consistency_bins[-1] = np.inf

train_features["late_consistency_quintile"] = pd.cut(
    train_features["late_momentum_consistency"],
    bins=consistency_bins,
    labels=["Q1", "Q2", "Q3", "Q4", "Q5"],
    include_lowest=True,
    duplicates="drop"
)

val_features["late_consistency_quintile"] = pd.cut(
    val_features["late_momentum_consistency"],
    bins=consistency_bins,
    labels=["Q1", "Q2", "Q3", "Q4", "Q5"],
    include_lowest=True,
    duplicates="drop"
)

train_consistency = pd.crosstab(
    train_features["late_consistency_quintile"],
    train_features["reod"],
    normalize="index"
)

val_consistency = pd.crosstab(
    val_features["late_consistency_quintile"],
    val_features["reod"],
    normalize="index"
)

print("TRAIN")
display(train_consistency)

print("VALIDATION")
display(val_consistency)

TRAIN


reod,-1,0,1
late_consistency_quintile,,,
Q1,0.291440,0.434753,0.273807
Q2,0.314145,0.384090,0.301765
Q3,0.325717,0.360416,0.313868
Q4,0.316895,0.370145,0.312960
Q5,0.249255,0.490390,0.260355


VALIDATION


reod,-1,0,1
late_consistency_quintile,,,
Q1,0.294908,0.440231,0.264861
Q2,0.318503,0.396405,0.285092
Q3,0.337714,0.366283,0.296003
Q4,0.321135,0.389852,0.289012
Q5,0.267287,0.513627,0.219086


### Late trend

Calculamos la pendiente de la trayectoria acumulada durante el bloque late.

In [16]:
def add_late_trend(df, late_cols):
    out = df.copy()

    late_returns = out[late_cols].to_numpy(dtype=float)
    trends = np.full(len(out), np.nan)

    for i, row in enumerate(late_returns):
        mask = ~np.isnan(row)

        if mask.sum() < 2:
            continue

        r = row[mask]

        # trayectoria acumulada dentro del bloque late
        path = np.cumsum(r)
        x = np.arange(len(path))

        trends[i] = np.polyfit(x, path, 1)[0]

    out["late_trend_bps"] = trends

    return out


late_cols = [f"r{i}" for i in range(36, 53)]

train_features = add_late_trend(train_features, late_cols)
val_features = add_late_trend(val_features, late_cols)

train_features["late_trend_bps"].describe(
    percentiles=[.25, .50, .75, .90, .95, .99]
)

count    658841.000000
mean         -0.222932
std           7.145821
min        -147.750500
25%          -2.193897
50%           0.000000
75%           1.975907
90%           4.975000
95%           7.892059
99%          18.710500
max         143.245260
Name: late_trend_bps, dtype: float64